# Vector Search Setup with Ollama Embeddings + Neo4j

This notebook:
1. Loads document chunks from shareholder letters
2. Creates embeddings using Ollama (nomic-embed-text)
3. Stores chunks + embeddings in Neo4j with vector index
4. Tests vector similarity search

**Prerequisites:**
- Ollama running with `nomic-embed-text` model pulled
- Neo4j AuraDB connection configured in `.env`
- Knowledge graph already populated (from previous notebook)

## 1. Setup & Imports

In [1]:
import os
from dotenv import load_dotenv
from tqdm import tqdm

from langchain_ollama import OllamaEmbeddings
from langchain_neo4j import Neo4jGraph, Neo4jVector
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
print("✅ Imports successful")

✅ Imports successful


## 2. Initialize Ollama Embeddings

In [2]:
# Initialize Ollama embeddings
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",  # 768 dimensions
)

# Test the embeddings
print("🧪 Testing embeddings...")
test_text = "Microsoft's cloud computing platform Azure"
test_vector = embeddings.embed_query(test_text)

print(f"✅ Embeddings working!")
print(f"   - Dimensions: {len(test_vector)}")
print(f"   - Sample values: [{test_vector[0]:.4f}, {test_vector[1]:.4f}, {test_vector[2]:.4f}, ...]")

🧪 Testing embeddings...
✅ Embeddings working!
   - Dimensions: 768
   - Sample values: [-0.0122, 0.0749, -0.1552, ...]


## 3. Connect to Neo4j

In [3]:
# Neo4j connection parameters
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

print("🔌 Connecting to Neo4j...")

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD
)

print("✅ Connected to Neo4j")

🔌 Connecting to Neo4j...
✅ Connected to Neo4j


## 4. Load and Chunk Documents

In [4]:
# Load shareholder letters
print("📂 Loading documents from ./data...")

loader = DirectoryLoader(
    './data',
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)

raw_docs = loader.load()
print(f"✅ Loaded {len(raw_docs)} documents")

for doc in raw_docs:
    filename = doc.metadata.get('source', 'unknown').split('/')[-1].split('\\')[-1]
    print(f"   - {filename}")

📂 Loading documents from ./data...
✅ Loaded 5 documents
   - 2020_shareholder_letter.txt
   - 2021_shareholder_letter.txt
   - 2022_shareholder_letter.txt
   - 2023_shareholder_letter.txt
   - 2024_shareholder_letter.txt


In [5]:
# Chunk documents for vector storage
# Using smaller chunks for better semantic retrieval
print("✂️ Chunking documents...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # Smaller chunks for vector search
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(raw_docs)

print(f"✅ Created {len(chunks)} chunks")
print(f"   - Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")

✂️ Chunking documents...
✅ Created 181 chunks
   - Average chunk size: 739 chars


In [6]:
# Preview a few chunks
print("📝 Sample chunks:")
for i, chunk in enumerate(chunks[:3]):
    source = chunk.metadata.get('source', 'unknown').split('/')[-1].split('\\')[-1]
    print(f"\n--- Chunk {i+1} (from {source}) ---")
    print(chunk.page_content[:200] + "...")

📝 Sample chunks:

--- Chunk 1 (from 2020_shareholder_letter.txt) ---
Dear shareholders, colleagues, customers, and partners:

While the start of a new decade typically brings hope, we quickly saw the world come to a near standstill in 2020, confronted by compounding cr...

--- Chunk 2 (from 2020_shareholder_letter.txt) ---
It is in times like these that our ability to stay true to Microsoft’s mission and corporate purpose is of the utmost importance. As a company, we are steadfast in our mission to empower every person ...

--- Chunk 3 (from 2020_shareholder_letter.txt) ---
I’m proud of how our ecosystem of customers and partners has stepped up over the past year to help people and organizations in every country use technology to be resilient and transform during the mos...


## 5. Create Vector Index in Neo4j

We'll store chunks as `DocumentChunk` nodes with:
- `text`: The chunk content
- `source`: Source file
- `embedding`: Vector embedding (768 dimensions)

In [7]:
# Check if we already have chunks
result = graph.query("MATCH (c:DocumentChunk) RETURN count(c) as count")
existing_count = result[0]['count'] if result else 0

print(f"📊 Existing DocumentChunk nodes: {existing_count}")

if existing_count > 0:
    print("\n⚠️ Chunks already exist. Options:")
    print("   1. Skip ingestion (use existing)")
    print("   2. Delete and re-ingest")

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownLabelWarning} {category: UNRECOGNIZED} {title: The provided label is not in the database.} {description: One of the labels in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing label name is: DocumentChunk)} {position: line: 1, column: 10, offset: 9} for query: 'MATCH (c:DocumentChunk) RETURN count(c) as count'


📊 Existing DocumentChunk nodes: 0


In [8]:
# Uncomment to clear existing chunks
CLEAR_EXISTING_CHUNKS = True  # Set to True to re-ingest

if CLEAR_EXISTING_CHUNKS and existing_count > 0:
    print("🗑️ Clearing existing DocumentChunk nodes...")
    graph.query("MATCH (c:DocumentChunk) DETACH DELETE c")
    print("✅ Cleared")

In [9]:
# Create vector index if it doesn't exist
print("📐 Creating vector index...")

# Drop existing index if present (to recreate with correct dimensions)
try:
    graph.query("DROP INDEX document_embeddings IF EXISTS")
except:
    pass

# Create vector index for DocumentChunk nodes
# nomic-embed-text produces 768-dimensional vectors
graph.query("""
    CREATE VECTOR INDEX document_embeddings IF NOT EXISTS
    FOR (c:DocumentChunk)
    ON c.embedding
    OPTIONS {
        indexConfig: {
            `vector.dimensions`: 768,
            `vector.similarity_function`: 'cosine'
        }
    }
""")

print("✅ Vector index created (768 dimensions, cosine similarity)")

📐 Creating vector index...
✅ Vector index created (768 dimensions, cosine similarity)


## 6. Ingest Chunks with Embeddings

In [10]:
# Ingest chunks with embeddings
print(f"📥 Ingesting {len(chunks)} chunks with embeddings...")
print("   (This may take a few minutes)")

batch_size = 10  # Process in batches

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i+batch_size]
    
    # Get texts for embedding
    texts = [chunk.page_content for chunk in batch]
    
    # Generate embeddings for batch
    batch_embeddings = embeddings.embed_documents(texts)
    
    # Create nodes in Neo4j
    for j, (chunk, embedding) in enumerate(zip(batch, batch_embeddings)):
        chunk_id = f"chunk_{i + j}"
        source = chunk.metadata.get('source', 'unknown').split('/')[-1].split('\\')[-1]
        
        # Create DocumentChunk node with embedding
        graph.query("""
            CREATE (c:DocumentChunk {
                id: $chunk_id,
                text: $text,
                source: $source,
                embedding: $embedding
            })
        """, params={
            "chunk_id": chunk_id,
            "text": chunk.page_content,
            "source": source,
            "embedding": embedding
        })

print(f"\n✅ Ingested {len(chunks)} chunks with embeddings!")

📥 Ingesting 181 chunks with embeddings...
   (This may take a few minutes)


100%|██████████████████████████████████████████████████████████████████████████████████| 19/19 [00:25<00:00,  1.32s/it]


✅ Ingested 181 chunks with embeddings!


## 7. Verify Ingestion

In [11]:
# Verify what's stored
print("📊 Verification:")

result = graph.query("MATCH (c:DocumentChunk) RETURN count(c) as count")
print(f"   - DocumentChunk nodes: {result[0]['count']}")

result = graph.query("""
    MATCH (c:DocumentChunk)
    RETURN c.source as source, count(*) as chunks
    ORDER BY source
""")
print("\n   Chunks per source:")
for row in result:
    print(f"   - {row['source']}: {row['chunks']} chunks")

📊 Verification:
   - DocumentChunk nodes: 181

   Chunks per source:
   - 2020_shareholder_letter.txt: 37 chunks
   - 2021_shareholder_letter.txt: 32 chunks
   - 2022_shareholder_letter.txt: 34 chunks
   - 2023_shareholder_letter.txt: 37 chunks
   - 2024_shareholder_letter.txt: 41 chunks


In [12]:
# Verify embeddings are stored
result = graph.query("""
    MATCH (c:DocumentChunk)
    WHERE c.embedding IS NOT NULL
    RETURN c.id as id, size(c.embedding) as dimensions
    LIMIT 3
""")

print("📊 Embedding verification:")
for row in result:
    print(f"   - {row['id']}: {row['dimensions']} dimensions")

📊 Embedding verification:
   - chunk_0: 768 dimensions
   - chunk_1: 768 dimensions
   - chunk_2: 768 dimensions


## 8. Test Vector Similarity Search

In [13]:
def vector_search(query: str, top_k: int = 5) -> list:
    """
    Perform vector similarity search in Neo4j.
    
    Args:
        query: Search query text
        top_k: Number of results to return
        
    Returns:
        List of matching chunks with scores
    """
    # Generate embedding for query
    query_embedding = embeddings.embed_query(query)
    
    # Search using Neo4j vector index
    results = graph.query("""
        CALL db.index.vector.queryNodes('document_embeddings', $top_k, $embedding)
        YIELD node, score
        RETURN node.id as id, node.text as text, node.source as source, score
        ORDER BY score DESC
    """, params={
        "embedding": query_embedding,
        "top_k": top_k
    })
    
    return results

In [14]:
# Test query 1: Strategic topic
print("🔍 Test Query 1: 'Microsoft's AI strategy and investments'")
print("=" * 60)

results = vector_search("Microsoft's AI strategy and investments", top_k=3)

for i, row in enumerate(results):
    print(f"\n📄 Result {i+1} (Score: {row['score']:.4f}) - {row['source']}")
    print(f"   {row['text'][:300]}...")

🔍 Test Query 1: 'Microsoft's AI strategy and investments'

📄 Result 1 (Score: 0.7987) - 2021_shareholder_letter.txt
   Care is the new currency for every leader, and we’ve built a new framework to help our managers strengthen their teams and deliver success through empowerment and accountability. Our managers strive to model our culture and values in their actions, to coach their teams to define objectives and adapt...

📄 Result 2 (Score: 0.7969) - 2022_shareholder_letter.txt
   Gaming

The big bets we have made across content, community, and cloud over the past few years continue to pay off. We’ve sold more Xbox Series S and Series X consoles life-to-date than any previous generation of Xbox, and with Xbox Cloud Gaming, we’re bringing games to entirely new endpoints. In th...

📄 Result 3 (Score: 0.7952) - 2024_shareholder_letter.txt
   Coles is generating 1.6 billion daily AI predictions across 850 Australian stores, ensuring every shopper finds what they need.
    Unilever is perform

In [15]:
# Test query 2: Financial topic
print("🔍 Test Query 2: 'cloud revenue growth Azure'")
print("=" * 60)

results = vector_search("cloud revenue growth Azure", top_k=3)

for i, row in enumerate(results):
    print(f"\n📄 Result {i+1} (Score: {row['score']:.4f}) - {row['source']}")
    print(f"   {row['text'][:300]}...")

🔍 Test Query 2: 'cloud revenue growth Azure'

📄 Result 1 (Score: 0.8644) - 2024_shareholder_letter.txt
   Infrastructure

This year, we expanded our cloud and AI capacity, announcing new investments across five continents. These are long-term assets to drive new growth for the next decade and beyond, and ensure communities around the world have access to the compute they need to drive economic growth in...

📄 Result 2 (Score: 0.8544) - 2023_shareholder_letter.txt
   To build on this progress, we remain convicted on three things: First, we will maintain our lead as the top commercial cloud while innovating in consumer categories, from gaming to professional social networks. Second, because we know that maximum enterprise value gets created during platform shifts...

📄 Result 3 (Score: 0.8443) - 2020_shareholder_letter.txt
   I’m proud of how our ecosystem of customers and partners has stepped up over the past year to help people and organizations in every country use technology to be re

In [16]:
# Test query 3: Risk topic
print("🔍 Test Query 3: 'cybersecurity threats and risks'")
print("=" * 60)

results = vector_search("cybersecurity threats and risks", top_k=3)

for i, row in enumerate(results):
    print(f"\n📄 Result {i+1} (Score: {row['score']:.4f}) - {row['source']}")
    print(f"   {row['text'][:300]}...")

🔍 Test Query 3: 'cybersecurity threats and risks'

📄 Result 1 (Score: 0.8359) - 2022_shareholder_letter.txt
   Security and digital safety are foundational to trust in today’s complex threat landscape. We analyze 43 trillion security signals daily and use the insights to inform increased protections. This year, we blocked 34.7 billion identity threats and 37 billion email threats. Over the past four years, w...

📄 Result 2 (Score: 0.8164) - 2023_shareholder_letter.txt
   The era of AI heightens the importance of cybersecurity, and we deepened our work across the private and public sectors to improve cyber-resilience. We’ve continued to support Ukraine in defending critical infrastructure, detecting and disrupting cyberattacks and cyberinfluence operations, and provi...

📄 Result 3 (Score: 0.8106) - 2024_shareholder_letter.txt
   Our Secure Future Initiative advances how we design, build, test, and operate our technology to ensure we deliver solutions that meet the highest possible stan

## 9. Summary

✅ **Vector search is now set up!**

Your Neo4j database now contains:
- **Knowledge Graph**: Organizations, Products, Strategies, etc. (from previous notebook)
- **Vector Index**: Document chunks with embeddings for semantic search

### Next Steps:
1. Update the retriever node in the agent to use vector search
2. Test the full agent workflow

### Example Cypher for Vector Search:
```cypher
// Find similar documents
CALL db.index.vector.queryNodes('document_embeddings', 5, $embedding)
YIELD node, score
RETURN node.text, node.source, score
```

In [17]:
# Check what's stored
result = graph.query("""
    MATCH (c:DocumentChunk) 
    RETURN count(c) as chunks,
           count(c.embedding) as with_embeddings
""")
print(f"✅ Chunks: {result[0]['chunks']}")
print(f"✅ With embeddings: {result[0]['with_embeddings']}")

✅ Chunks: 181
✅ With embeddings: 181


In [18]:
# Test a quick search
results = vector_search("Microsoft cloud strategy", top_k=2)
for r in results:
    print(f"\nScore: {r['score']:.3f}")
    print(f"Text: {r['text'][:150]}...")


Score: 0.840
Text: And with Windows 365, we are creating a new category: the cloud PC. By bringing the operating system to the cloud, we’re enabling organizations to str...

Score: 0.823
Text: Our industry clouds bring together capabilities across the Microsoft Cloud with industry-specific customizations to help organizations improve time to...
